# Scattering Simulator CK: Mumax OVF Magnetic Pattern

This notebook mirrors the CK scattering workflow, but the magnetic texture is read from a Mumax/OOMMF OVF file instead of being generated procedurally.

The important extra steps are:

1. choose a Mumax `.ovf` file from the external data folder, `DATA_ROOT / "Data" / "mumax_files"`;
2. parse the OVF mesh header and binary vector field `(m_x, m_y, m_z)`;
3. ask the user for a material recipe and check that the number of magneto-optic layers matches `znodes`;
4. interpolate the Mumax vector field onto the simulator sample grid;
5. apply the holography mask, propagate the wavefield, and optionally export the result in the standard HDF5 layout.

The OVF file is interpreted as a physical magnetization volume. `xstepsize`, `ystepsize`, and `zstepsize` are in metres; the simulator interpolation uses those physical lengths, not pixel labels.

## 1. Imports

In [ ]:
# General libraries
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import os
import warnings

# Keep Matplotlib/fontconfig caches inside the writable project tree. If these
# caches are unavailable, first imports/plots can randomly take much longer.
_LOCAL_CACHE = Path.cwd() / ".notebook_cache"
_MPL_CACHE = _LOCAL_CACHE / "matplotlib"
_XDG_CACHE = _LOCAL_CACHE / "xdg"
_MPL_CACHE.mkdir(parents=True, exist_ok=True)
_XDG_CACHE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(_MPL_CACHE))
os.environ.setdefault("XDG_CACHE_HOME", str(_XDG_CACHE))

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator

%matplotlib widget
plt.rcParams["figure.constrained_layout.use"] = True

# Project imports
from fomocid import DATA_ROOT
from scattering_calculator.interactive.interactive_widgets import cimshow
from scattering_calculator.simulation_pipelines import simulation_configuration
from scattering_calculator.simulation_pipelines import HologramPipeline, HologramPipelineConfig, HologramPipelineRanges
from scattering_calculator.utils.mumax import list_ovf_files, read_mumax_ovf


## 2. OVF Reader

Run this cell first. Set `MUMAX_FILE_NAME` to choose the OVF file. By default the folder listing is skipped, then the header is parsed and a lazy memory-mapped view of the vector field is returned. The cell should be fast even for large Mumax files because the full array is not copied into RAM. Later plotting/interpolation cells read only the data they actually need.


In [ ]:
MUMAX_FOLDER = DATA_ROOT / "Data" / "mumax_files"
MUMAX_FILE_NAME = "0000.ovf"  # Edit this value to choose another OVF file.
LIST_AVAILABLE_OVF = False  # Set True only when you want to scan and list the folder.

import time

_t0 = time.perf_counter()
print(f"Mumax folder: {MUMAX_FOLDER}")
if LIST_AVAILABLE_OVF:
    available_ovf = list_ovf_files(MUMAX_FOLDER)
    print("Available OVF files:")
    for candidate in available_ovf:
        print(f" - {candidate.name} ({candidate.stat().st_size / 1024**2:.1f} MiB)")
    if not available_ovf:
        print("No OVF files found yet. Put files in DATA_ROOT / 'Data' / 'mumax_files'.")
else:
    print("Skipping folder scan; set LIST_AVAILABLE_OVF = True to list available OVF files.")
print(f"Folder/setup time: {time.perf_counter() - _t0:.3f} s")

mumax_path = MUMAX_FOLDER / MUMAX_FILE_NAME
_t0 = time.perf_counter()
mumax = read_mumax_ovf(mumax_path, mmap=True)
print(f"OVF header + memmap setup time: {time.perf_counter() - _t0:.3f} s")
print(f"Loaded lazy array type: {type(mumax.magnetization).__name__}, dtype={mumax.magnetization.dtype}")
print(f"Shape (z, y, x, vector): {mumax.magnetization.shape}")
print(f"Cell size: dx={mumax.dx:.3g} m, dy={mumax.dy:.3g} m, dz={mumax.dz:.3g} m")
print(f"Physical size: x={mumax.size_x:.3g} m, y={mumax.size_y:.3g} m, z={mumax.size_z:.3g} m")
print(f"Magnetic sublayers required by this file: {mumax.znodes}")


## 3. Inspect The Mumax Pattern

This cell gives a quick visual check of the Mumax magnetization that was loaded from the OVF file. It shows the z-averaged out-of-plane magnetization, the middle magnetic slice, and the local magnetization norm.


In [ ]:
mid_z = mumax.znodes // 2
mz_projection = np.asarray(mumax.magnetization[:, :, :, 2]).mean(axis=0)
mz_middle = np.asarray(mumax.magnetization[mid_z, :, :, 2])
norm_middle = np.linalg.norm(np.asarray(mumax.magnetization[mid_z, :, :, :]), axis=-1)

extent_nm = [
    float(mumax.header["xmin"]) * 1e9,
    float(mumax.header["xmax"]) * 1e9,
    float(mumax.header["ymin"]) * 1e9,
    float(mumax.header["ymax"]) * 1e9,
]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharex=True, sharey=True)
images = [
    (mz_projection, "mean z projection of $m_z$"),
    (mz_middle, f"middle layer $m_z$ (z index {mid_z})"),
    (norm_middle, f"|m|, z index {mid_z}"),
]
for ax, (image, title) in zip(axes, images):
    im = ax.imshow(image, origin="lower", cmap="coolwarm", vmin=-1, vmax=1, extent=extent_nm)
    ax.set_title(title)
    ax.set_xlabel("x (nm)")
axes[0].set_ylabel("y (nm)")
fig.colorbar(im, ax=axes, label="magnetization")
plt.show()


## 4. Experimental Geometry

These cells are the same kind of setup used in the CK notebook. Change the detector, beamstop, and flux parameters here before building the sample.

In [ ]:
# X-ray source
xrayconfig = simulation_configuration.XRayConfig(
    energy=787.9,        # eV
    pol="CR",           # initial polarization; overwritten in the CR/CL loop
    photon_flux=1e10,    # photons per second
    coherence_length=(20e-6, 20e-6),
)
xrayconfig.setup()

# Detector and beamstop
detector_pixel_size = 20e-6
ndet = 512
detector_pixel_shape = (ndet, ndet)
detector_distance = 0.10
detector_center = (ndet // 2, ndet // 2)

detector_params = {
    "readout_noise_average": 50,
    "readout_noise_sigma": 3,
    "detector_threshold": 64e3,
    "counts_per_photon": 180,
    "quantum_efficiency": 0.85,
}
measurement_config = {
    "number_frames": 100,
    "max_counts_per_image": 63.5e3,
    "exposure_time": 5.0,
}
artifacts_config = {
    "sigma_photon": 0.75,
    "photon_n_classes": 1,
    "photon_n_variants": 30,
    "photon_kernel_size": 9,
    "photon_irregularity": 2.0,
    "regenerate_photon_kernels": True,
}

beamstop_distance = 0.001
beamstop_center = np.array(detector_pixel_shape) // 2
beamstop_method = "circular"
beamstop_config = simulation_configuration.BeamstopConfig(
    bs_method=beamstop_method,
    bs_detector_distance=beamstop_distance,
    bs_center=beamstop_center,
    bs_config={
        "radius": 0.5e-3,
        "angle": np.pi / 6,
        "sigma": 0.01e-3,
        "ellipticity": (0.9, 1.1),
        "roughness": 0.05,
        "roughness_modes": (3, 9),
        "wire_width": 0.05e-3,
        "wire_bend": 0.1e-3,
        "antialias": 2,
    },
)

detectorconfig = simulation_configuration.DetectorConfig(
    shape=detector_pixel_shape,
    pixel_size=detector_pixel_size,
    sample_to_detector_distance=detector_distance,
    detector_center=detector_center,
    detector_params=detector_params,
    measurement_config=measurement_config,
    artifacts_config=artifacts_config,
    beamstop_config=beamstop_config,
)
detectorconfig.setup()
oversampling = 2
real_space_pixel_size = detectorconfig.calc_realspace_resolution(xrayconfig.beam_params) / oversampling
print(f"Sample-plane pixel size with oversampling={oversampling}: {real_space_pixel_size * 1e9:.3f} nm")


## 5. User Recipe And Mumax Compatibility Check

Edit `recipe` so the magnetic part of the multilayer has exactly one propagated magnetic layer per Mumax z-cell. Slash-separated recipe terms become separate propagated layers; adjacent terms without a slash are combined into one effective layer.

Example of an incompatible recipe: if `znodes=20`, `[Pt(3)/Co(2)]x3/SiN(200)/Au(1999)` contains only three Co-like magnetic layers, not twenty.

In [ ]:
# Edit this recipe after reading the OVF report above.
# This example is deliberately written as one magnetic layer per Mumax z cell.
magnetic_layer_material = "Co"
recipe = f"[Au(50)/Cr(5)]x20/SiN(100)/[{magnetic_layer_material}({mumax.zstepsize * 1e9:.6g})]x{mumax.znodes}"

sample_shape = np.array(
    [0, oversampling * detectorconfig.shape[0], oversampling * detectorconfig.shape[1]],
    dtype=int,
)

sampleconfig = simulation_configuration.SampleConfig(
    recipe=recipe,
    sample_shape=sample_shape,
    real_space_pixel_size=real_space_pixel_size,
    xray_config=xrayconfig,
    sample_name="mumax_ovf_sample",
)
sampleconfig.setup()

layer_names = np.asarray(sampleconfig.sample_structure.layer_names)
layer_thicknesses = np.asarray(sampleconfig.sample_structure.layer_thicknesses, dtype=float)
dt = np.asarray(sampleconfig.sample_structure.dielectric_tensors)
eps_mz = dt[:, 1]
eps_xy = dt[:, 2]
magnetic_layer_mask = np.any(np.abs(eps_mz) > 1e-14, axis=(1, 2)) | np.any(np.abs(eps_xy) > 1e-14, axis=(1, 2))
magnetic_layer_indices = np.flatnonzero(magnetic_layer_mask)

print("Recipe:", recipe)
print(f"Total propagated layers in recipe: {len(layer_names)}")
print(f"Magneto-optic propagated layers in recipe: {len(magnetic_layer_indices)}")
print(f"Mumax z nodes: {mumax.znodes}")

if len(magnetic_layer_indices) != mumax.znodes:
    raise ValueError(
        "Mumax/recipe mismatch: "
        f"the OVF file has znodes={mumax.znodes}, but the recipe produced "
        f"{len(magnetic_layer_indices)} magneto-optic propagated layers. "
        "Use slash-separated magnetic layers or a repeated block with one magnetic propagated layer per Mumax z cell."
    )

magnetic_thicknesses = layer_thicknesses[magnetic_layer_indices]
relative_thickness_error = np.max(np.abs(magnetic_thicknesses - mumax.zstepsize) / max(mumax.zstepsize, 1e-30))
print(f"Mumax zstepsize: {mumax.zstepsize * 1e9:.6g} nm")
print(
    "Magnetic recipe layer thickness range: "
    f"{magnetic_thicknesses.min() * 1e9:.6g} to {magnetic_thicknesses.max() * 1e9:.6g} nm"
)
if relative_thickness_error > 0.05:
    warnings.warn(
        "The magnetic layer thicknesses differ from the Mumax zstepsize by more than 5%. "
        "The layer count is compatible, but the physical z assignment may not be what you intended.",
        RuntimeWarning,
    )

print("First magnetic layer assignments:")
for ovf_z, layer_idx in list(zip(range(mumax.znodes), magnetic_layer_indices))[:10]:
    print(
        f"  OVF z {ovf_z:03d} -> sample layer {layer_idx:03d} "
        f"{layer_names[layer_idx]} ({layer_thicknesses[layer_idx] * 1e9:.4g} nm)"
    )
if mumax.znodes > 10:
    print("  ...")


## 6. Interpolate Mumax Magnetization Onto The Scattering Grid

The Mumax coordinate system is recentered on the sample origin, then interpolated onto the simulator lateral grid. Mumax simulations usually use periodic boundary conditions in the lateral directions, so this notebook wraps the OVF field periodically in `x` and `y`. If the Mumax box is smaller than the object-hole region or the full scattering grid, the texture is tiled instead of being filled with zero magnetization.

The resulting `magnetization` array has shape `(sample_Nz, sample_Ny, sample_Nx, 3)`. Only magneto-optic recipe layers receive the Mumax z slices; non-magnetic layers are left at zero.


In [ ]:
def sample_lateral_coordinates(shape_yx, pixel_size):
    ny, nx = map(int, shape_yx)
    y = (np.arange(ny) - ny / 2 + 0.5) * pixel_size
    x = (np.arange(nx) - nx / 2 + 0.5) * pixel_size
    return y, x


def mumax_cell_centers(meta):
    x = float(meta["xmin"]) + (np.arange(int(meta["xnodes"])) + 0.5) * float(meta["xstepsize"])
    y = float(meta["ymin"]) + (np.arange(int(meta["ynodes"])) + 0.5) * float(meta["ystepsize"])
    z = float(meta["zmin"]) + (np.arange(int(meta["znodes"])) + 0.5) * float(meta["zstepsize"])
    # Recenter x/y so the Mumax pattern is placed around the optical/sample origin.
    x = x - 0.5 * (float(meta["xmin"]) + float(meta["xmax"]))
    y = y - 0.5 * (float(meta["ymin"]) + float(meta["ymax"]))
    return z, y, x


def wrap_periodic_coordinates(coords, centers):
    step = float(np.mean(np.diff(centers)))
    period = centers.size * step
    return ((coords - centers[0]) % period) + centers[0]


def extend_periodic_grid(values):
    """Add one wrapped row/column so interpolation is continuous at boundaries."""
    return np.pad(values, ((0, 1), (0, 1)), mode="wrap")


def interpolate_mumax_to_sample(mumax, sample_shape, sample_pixel_size, magnetic_layer_indices):
    _, y_mumax, x_mumax = mumax_cell_centers(mumax.header)
    sample_y, sample_x = sample_lateral_coordinates(sample_shape[1:], sample_pixel_size)

    y_step = float(np.mean(np.diff(y_mumax)))
    x_step = float(np.mean(np.diff(x_mumax)))
    y_period = y_mumax.size * y_step
    x_period = x_mumax.size * x_step
    y_extended = np.concatenate([y_mumax, [y_mumax[-1] + y_step]])
    x_extended = np.concatenate([x_mumax, [x_mumax[-1] + x_step]])

    wrapped_y = wrap_periodic_coordinates(sample_y, y_mumax)
    wrapped_x = wrap_periodic_coordinates(sample_x, x_mumax)
    yy, xx = np.meshgrid(wrapped_y, wrapped_x, indexing="ij")
    query = np.column_stack([yy.ravel(), xx.ravel()])

    out = np.zeros((*sample_shape, 3), dtype=np.float32)
    for ovf_z, layer_idx in enumerate(magnetic_layer_indices):
        for component in range(3):
            periodic_values = extend_periodic_grid(mumax.magnetization[ovf_z, :, :, component])
            interpolator = RegularGridInterpolator(
                (y_extended, x_extended),
                periodic_values,
                bounds_error=False,
                fill_value=None,
            )
            out[layer_idx, :, :, component] = interpolator(query).reshape(sample_shape[1:])

    norm = np.linalg.norm(out, axis=-1)
    too_large = norm > 1.0
    if np.any(too_large):
        out[too_large] /= norm[too_large, None]
    print(
        "Periodically tiled Mumax field over the full sample grid "
        f"(unit cell: {x_period * 1e9:.3g} nm x {y_period * 1e9:.3g} nm)."
    )
    return out


magnetization = interpolate_mumax_to_sample(
    mumax,
    sampleconfig.sample_structure.sample_shape,
    sampleconfig.sample_structure.real_space_pixel_size,
    magnetic_layer_indices,
)
sampleconfig.assign_magnetic_pattern(magnetization)

mz_layers = magnetization[magnetic_layer_indices, :, :, 2]
magnetic_pattern = np.mean(mz_layers, axis=0)
coverage = np.ones(sampleconfig.sample_structure.sample_shape[1:], dtype=bool)
print("Interpolated magnetization shape:", magnetization.shape)
print("Mumax periodic tiling covers 100.000% of the scattering grid by construction.")


## 7. Plot The Interpolated Magnetic Pattern

These plots show the actual Mumax magnetization after interpolation onto the scattering sample grid, before the holography mask is applied.

The first figure is a compact overview: mean `m_z`, middle-layer `m_z`, mean `m_x`, and the periodic tiling mask. The second figure shows several xy planes through the magnetic stack so layer-to-layer changes are visible. The third figure follows the common Mumax visualization style: `m_z` is the background image, while in-plane `(m_x, m_y)` components are drawn as arrows colored by their in-plane angle. A separate hue/saturation panel shows the in-plane direction as hue and in-plane magnitude as saturation; black means no magnetic vector is present. Because the OVF is wrapped periodically, the tiling mask should cover the full plotted grid.


In [ ]:
sample_extent_nm = [
    -0.5 * sample_shape[2] * real_space_pixel_size * 1e9,
    +0.5 * sample_shape[2] * real_space_pixel_size * 1e9,
    -0.5 * sample_shape[1] * real_space_pixel_size * 1e9,
    +0.5 * sample_shape[1] * real_space_pixel_size * 1e9,
]

mx_layers = magnetization[magnetic_layer_indices, :, :, 0]
my_layers = magnetization[magnetic_layer_indices, :, :, 1]
mean_mx = np.mean(mx_layers, axis=0)
mean_my = np.mean(my_layers, axis=0)


def magnetic_xy_rgb(mx, my, mz):
    """RGB map inspired by Mumax: hue = in-plane angle, saturation = |m_xy|."""
    in_plane = np.sqrt(mx**2 + my**2)
    norm = np.sqrt(mx**2 + my**2 + mz**2)
    hue = (np.arctan2(my, mx) + np.pi) / (2 * np.pi)
    saturation = np.clip(in_plane, 0, 1)
    value = np.clip(norm, 0, 1)
    hsv = np.stack([hue, saturation, value], axis=-1)
    rgb = plt.cm.hsv(hue)[..., :3]
    # Blend the hue toward grayscale when the in-plane component is weak.
    gray = np.repeat(value[..., None], 3, axis=-1)
    return gray * (1 - saturation[..., None]) + rgb * saturation[..., None] * value[..., None]


def add_inplane_quiver(ax, mx, my, *, number_arrows_x=24, threshold=0.08):
    ny, nx = mx.shape
    step = max(1, int(round(nx / number_arrows_x)))
    y_idx = np.arange(step // 2, ny, step)
    x_idx = np.arange(step // 2, nx, step)
    xx_nm = sample_extent_nm[0] + (x_idx + 0.5) * real_space_pixel_size * 1e9
    yy_nm = sample_extent_nm[2] + (y_idx + 0.5) * real_space_pixel_size * 1e9
    X, Y = np.meshgrid(xx_nm, yy_nm)
    U = mx[np.ix_(y_idx, x_idx)]
    V = my[np.ix_(y_idx, x_idx)]
    magnitude = np.sqrt(U**2 + V**2)
    mask = magnitude < threshold
    angle = np.arctan2(V, U)
    ax.quiver(
        X,
        Y,
        np.ma.masked_where(mask, U),
        np.ma.masked_where(mask, V),
        np.ma.masked_where(mask, angle),
        cmap="hsv",
        clim=(-np.pi, np.pi),
        pivot="mid",
        angles="xy",
        scale_units="xy",
        scale=1.0 / (0.55 * step * real_space_pixel_size * 1e9),
        width=0.004,
    )


fig, axes = plt.subplots(1, 4, figsize=(15, 3.8), sharex=True, sharey=True)
plot_items = [
    (magnetic_pattern, "mean interpolated $m_z$", "coolwarm", -1, 1),
    (mz_layers[mumax.znodes // 2], "middle Mumax layer $m_z$", "coolwarm", -1, 1),
    (mean_mx, "mean interpolated $m_x$", "coolwarm", -1, 1),
    (coverage.astype(float), "Periodic tiling mask", "gray", 0, 1),
]
for ax, (image, title, cmap, vmin, vmax) in zip(axes, plot_items):
    im = ax.imshow(image, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax, extent=sample_extent_nm)
    ax.set_title(title)
    ax.set_xlabel("x (nm)")
axes[0].set_ylabel("y (nm)")
fig.colorbar(im, ax=axes, label="value")
plt.show()

slice_positions = np.unique(np.round(np.linspace(0, len(magnetic_layer_indices) - 1, min(5, len(magnetic_layer_indices)))).astype(int))
fig, axes = plt.subplots(2, len(slice_positions), figsize=(3.2 * len(slice_positions), 6.2), sharex=True, sharey=True)
if len(slice_positions) == 1:
    axes = axes[:, None]
for col, stack_idx in enumerate(slice_positions):
    sample_layer = magnetic_layer_indices[stack_idx]
    mz = mz_layers[stack_idx]
    mx = mx_layers[stack_idx]
    axes[0, col].imshow(mz, origin="lower", cmap="coolwarm", vmin=-1, vmax=1, extent=sample_extent_nm)
    axes[0, col].set_title(f"$m_z$, OVF z {stack_idx}")
    axes[0, col].set_xlabel("x (nm)")
    axes[0, col].set_ylabel("y (nm)")
    axes[1, col].imshow(mx, origin="lower", cmap="coolwarm", vmin=-1, vmax=1, extent=sample_extent_nm)
    axes[1, col].set_title(f"$m_x$, OVF z {stack_idx}")
    axes[1, col].set_xlabel("x (nm)")
for ax in axes[:, 0]:
    ax.set_ylabel("y (nm)")
fig.colorbar(axes[0, -1].images[0], ax=axes[0, :], label="$m_z$")
fig.colorbar(axes[1, -1].images[0], ax=axes[1, :], label="$m_x$")
plt.show()

middle_idx = len(magnetic_layer_indices) // 2
middle_mx = mx_layers[middle_idx]
middle_my = my_layers[middle_idx]
middle_mz = mz_layers[middle_idx]
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0), sharex=True, sharey=True)
axes[0].imshow(middle_mz, origin="lower", cmap="gray", vmin=-1, vmax=1, extent=sample_extent_nm)
add_inplane_quiver(axes[0], middle_mx, middle_my)
axes[0].set_title("middle xy plane: $m_z$ + in-plane arrows")
axes[1].imshow(magnetic_xy_rgb(middle_mx, middle_my, middle_mz), origin="lower", extent=sample_extent_nm)
axes[1].set_title("middle xy plane: hue = angle, saturation = $|m_{xy}|$")
axes[2].imshow(middle_mx, origin="lower", cmap="coolwarm", vmin=-1, vmax=1, extent=sample_extent_nm)
axes[2].set_title("middle xy plane: $m_x$")
for ax in axes:
    ax.set_xlabel("x (nm)")
axes[0].set_ylabel("y (nm)")
plt.show()


## 8. Holography Mask

Define the object hole and reference holes exactly as in the CK notebook. The magnetic contrast will only be visible through the object hole because the dielectric tensor is built together with the aperture mask.

In [ ]:
apertures_radius = [700e-9, 10e-9, 20e-9]
apertures_types = ["OH", "RH", "RH"]
apertures_centers = [(0.0, 0.0), (2.0e-6, -1.55e-6), (1.67e-6, 2.08e-6)]
apertures_sigma = [1.5e-9, 2.0e-9, 0.5e-9]
apertures_angle = [0.0, 0.0, 0.0]
apertures_ellipticity = [1.0, 1.0, 1.0]
apertures_top_radius_factor = [1.5, 4.0, 3.0]
apertures_roughness = [0.0, 0.02, 0.02]
apertures_roughness_modes = [(0, 0), (3, 9), (3, 9)]
apertures_seed = [1, 2, 3]

try:
    membrane_index = list(sampleconfig.sample_structure.layer_names).index("SiN")
except ValueError:
    raise ValueError("This notebook expects the recipe to contain a SiN membrane layer for the OH depth.")

thickness_OH = np.sum(sampleconfig.sample_structure.layer_thicknesses[:membrane_index])
aperture_taper_depth = np.sum(
    sampleconfig.sample_structure.layer_thicknesses[: max(0, membrane_index - 2)]
)

front_aperture_config = simulation_configuration.FrontApertureConfig(
    aperture_method="FTH_circular",
    aperture_shape=sample_shape,
    real_space_pixel_size=sampleconfig.sample_structure.real_space_pixel_size,
    aperture_thicknesses=sampleconfig.sample_structure.layer_thicknesses,
    aperture_config={
        "apertures_type": apertures_types,
        "apertures_radius": apertures_radius,
        "apertures_center": apertures_centers,
        "apertures_sigma": apertures_sigma,
        "apertures_angle": apertures_angle,
        "apertures_ellipticity": apertures_ellipticity,
        "apertures_roughness": apertures_roughness,
        "apertures_roughness_modes": apertures_roughness_modes,
        "apertures_seed": apertures_seed,
        "apertures_top_radius_factor": apertures_top_radius_factor,
        "thickness_OH": thickness_OH,
        "aperture_taper_depth": aperture_taper_depth,
    },
)
front_aperture_config.setup()
sampleconfig.assign_aperture_mask(front_aperture_config.return_aperture())

oh_mask = front_aperture_config.create_supportmask(
    output_shape=sample_shape[1:],
    output_pixel_size=sampleconfig.sample_structure.real_space_pixel_size,
    aperture_types=("OH",),
)
magnetic_pattern_oh = magnetic_pattern * oh_mask

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharex=True, sharey=True)
items = [
    (np.mean(sampleconfig.sample_structure.mask, axis=0), "depth-averaged aperture mask", "gray", 0, 1),
    (oh_mask, "OH support mask", "gray", 0, 1),
    (magnetic_pattern_oh, "$m_z$ visible through OH", "coolwarm", -1, 1),
]
for ax, (image, title, cmap, vmin, vmax) in zip(axes, items):
    im = ax.imshow(image, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax, extent=sample_extent_nm)
    ax.set_title(title)
    ax.set_xlabel("x (nm)")
axes[0].set_ylabel("y (nm)")
fig.colorbar(im, ax=axes, label="value")
plt.show()


Now combine the material refractive indices, Mumax magnetization stack, and holography mask into the optical interaction used for propagation. Jones propagation builds a compact dielectric tensor. Scalar propagation skips the tensor and uses the refractive-index channels `[n_total, n_circ, n_lin]` directly. With lazy Scalar mode, aperture ROI patches are computed layer by layer during propagation. Jones and Scalar use the same free-space convention: multislice FFT crops use `exp(-1j * kz * dz)`, and ROI pixels outside the crop keep the `kz = k0` plane-wave baseline `exp(-1j * k0 * dz)`.


In [ ]:
propagator_method = "Jones"  # Use "Scalar" to skip dielectric tensors and use refractive-index channels directly.
scalar_refractive_index_lazy = True  # Scalar: compute refractive-index ROI patches layer by layer instead of precomputing the whole compact stack.
if propagator_method == "Jones":
    sampleconfig.sample_structure.calculate_final_dielectric_tensor(
        use_aperture_roi=True,
        compact=True,
    )
    print("Final dielectric tensor representation:", type(sampleconfig.sample_structure.final_dielectric_tensor).__name__)
    print("Aperture support ROIs:", getattr(sampleconfig.sample_structure.final_dielectric_tensor, "aperture_support_regions", None))
elif propagator_method == "Scalar":
    print("Scalar propagation selected: skipping dielectric tensor construction.")
    if scalar_refractive_index_lazy:
        print("Scalar ROI patches will be built layer by layer during propagation.")
else:
    raise ValueError(f"Unknown propagator_method: {propagator_method!r}")


## 10. Illumination

In [ ]:
illumination_function = "gaussian"
illumination_center = (0.0, 0.0)
illumination_focus_distance = 10e-3
illumination_fwhm = 2.0e-6
illumination_alpha_beam = (0.0, 0.0)  # rad, (alpha_y, alpha_x)

illuminationconfig = simulation_configuration.IlluminationConfig(
    XRayConfig=xrayconfig,
    shape=sample_shape[-2:],
    real_space_pixel_size=real_space_pixel_size,
    illumination_function=illumination_function,
    illumination_config={
        "center": illumination_center,
        "distance": illumination_focus_distance,
        "fwhm": illumination_fwhm,
        "alpha_beam": illumination_alpha_beam,
    },
)
illuminationconfig.setup()
illuminationconfig.visualize_illumination()


## 11. Hologram Computation

In [ ]:
hologram_config = simulation_configuration.HologramConfig(
    sample_x=sampleconfig.sample_structure.x,
    sample_y=sampleconfig.sample_structure.y,
    detector_layout=detectorconfig.detector_layout,
)

propagate = True
jones_apply_zero_order_phase = True  # Jones and Scalar no-FFT modes keep the exp(-1j*k0*dz) phase between slices.
multislice_propagation_roi = True
multislice_propagation_roi_padding_px = 64
multislice_propagation_roi_merge_overlaps = True
propagation_padding_px = 128
propagation_padding_mode = "edge"
propagation_absorber_width_px = 64
propagation_absorber_strength = 6.0
propagation_absorber_profile = "cosine"

for i, polarization in enumerate(["CR", "CL"]):
    illuminationconfig.update_polarization(polarization)
    samplepropagationconfig = simulation_configuration.SamplePropagatorConfig(
        SampleConfig=sampleconfig,
        IlluminationConfig=illuminationconfig,
        propagator_method=propagator_method,
        propagator_config={
            "propagate": propagate,
            "jones_apply_zero_order_phase": jones_apply_zero_order_phase,
            "scalar_refractive_index_lazy": scalar_refractive_index_lazy,
            "propagation_padding_px": propagation_padding_px,
            "propagation_padding_mode": propagation_padding_mode,
            "propagation_absorber_width_px": propagation_absorber_width_px,
            "propagation_absorber_strength": propagation_absorber_strength,
            "propagation_absorber_profile": propagation_absorber_profile,
            "multislice_propagation_roi": multislice_propagation_roi,
            "multislice_propagation_roi_padding_px": multislice_propagation_roi_padding_px,
            "multislice_propagation_roi_merge_overlaps": multislice_propagation_roi_merge_overlaps,
        },
    )
    samplepropagationconfig.setup()
    detectorconfig.assign_propagated_wavefront(samplepropagationconfig)
    detectorconfig.detect_hologram()

    hologram_config.add_exit_waves({polarization: samplepropagationconfig.return_scalar_wavefield()})
    hologram_config.add_holograms({polarization: detectorconfig.return_ideal_hologram()}, source="ideal")
    hologram_config.add_holograms({polarization: detectorconfig.return_detected_hologram()}, source="detected")

hologram_config.compute_differences()
hologram_config.compute_sums()
hologram_config.compute_reconstructions()
print("Computed CR/CL holograms, helicity differences/sums, and FTH reconstructions.")


## 12. Inspect Outputs

This section checks the simulated exit waves, holograms, and FTH reconstructions. The helicity difference stored by `HologramConfig` is `CR - CL`; for the exit-wave view below we also plot `CL - CR`, which is often the more intuitive sign convention when comparing directly against the Mumax magnetization.


In [ ]:
def first_frame(array):
    array = np.asarray(array)
    return array[0] if array.ndim == 3 else array


def show_image(ax, image, title, cmap, *, vmin=None, vmax=None, log=False):
    image = np.asarray(image)
    if log:
        image = np.log10(np.maximum(image, 1e-12))
    if vmin is None or vmax is None:
        finite = image[np.isfinite(image)]
        if finite.size:
            lo, hi = np.nanpercentile(finite, [1, 99])
            vmin = lo if vmin is None else vmin
            vmax = hi if vmax is None else vmax
    im = ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_axis_off()
    return im


fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for row, key in enumerate(["CR", "CL"]):
    exit_wave = first_frame(hologram_config.exit_waves[key])
    holo = first_frame(hologram_config.ideal_holograms[key])
    amp = np.abs(exit_wave)
    phase = np.angle(exit_wave)
    show_image(axes[row, 0], amp, f"{key} exit-wave amplitude", "magma")
    show_image(axes[row, 1], phase, f"{key} exit-wave phase", "twilight", vmin=-np.pi, vmax=np.pi)
    show_image(axes[row, 2], holo, f"{key} ideal hologram log10", "viridis", log=True)
plt.show()

exit_wave_cl_minus_cr = first_frame(hologram_config.exit_waves["CL"]) - first_frame(hologram_config.exit_waves["CR"])
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
show_image(axes[0], np.abs(exit_wave_cl_minus_cr), "CL - CR exit-wave amplitude", "magma")
show_image(axes[1], np.angle(exit_wave_cl_minus_cr), "CL - CR exit-wave phase", "twilight", vmin=-np.pi, vmax=np.pi)
show_image(axes[2], np.real(exit_wave_cl_minus_cr), "CL - CR exit-wave real", "coolwarm")
show_image(axes[3], np.imag(exit_wave_cl_minus_cr), "CL - CR exit-wave imaginary", "coolwarm")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
show_image(axes[0], first_frame(hologram_config.ideal_holograms["diff"]), "CR - CL ideal hologram", "coolwarm")
show_image(axes[1], first_frame(hologram_config.ideal_holograms["sum"]), "CR + CL ideal hologram log10", "viridis", log=True)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
show_image(axes[0], first_frame(hologram_config.detected_holograms["diff"]), "CR - CL detected hologram", "coolwarm")
show_image(axes[1], first_frame(hologram_config.detected_holograms["sum"]), "CR + CL detected hologram log10", "viridis", log=True)
plt.show()

ideal_diff_reconstruction = first_frame(hologram_config.reconstructions["ideal"]["diff"])
detected_diff_reconstruction = first_frame(hologram_config.reconstructions["detected"]["diff"])

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for row, (source_name, reconstruction) in enumerate([
    ("ideal", ideal_diff_reconstruction),
    ("detected", detected_diff_reconstruction),
]):
    show_image(axes[row, 0], np.abs(reconstruction), f"{source_name} diff FTH amplitude", "inferno")
    show_image(axes[row, 1], np.angle(reconstruction), f"{source_name} diff FTH phase", "twilight", vmin=-np.pi, vmax=np.pi)
    show_image(axes[row, 2], np.real(reconstruction), f"{source_name} diff FTH real", "coolwarm")
plt.show()


## 13. Optional HDF5 Export

The export uses the same HDF5 layout as the pipeline notebooks. The small metadata adapter below records the OVF source and mesh fields under `metadata/sample/magnetic_pattern/`.

In [ ]:
class MumaxPatternMetadata:
    def __init__(self, mumax, magnetic_layer_indices):
        self.pattern_type_method = "mumax_ovf"
        self.shape = mumax.magnetization.shape[:3]
        self.real_space_pixel_size = (mumax.ystepsize, mumax.xstepsize)
        self.pattern_config = {
            "source_file": str(mumax.path),
            "xnodes": mumax.xnodes,
            "ynodes": mumax.ynodes,
            "znodes": mumax.znodes,
            "xstepsize_m": mumax.xstepsize,
            "ystepsize_m": mumax.ystepsize,
            "zstepsize_m": mumax.zstepsize,
            "assigned_sample_layers": np.asarray(magnetic_layer_indices, dtype=int),
        }

    def get_metadata(self, prefix=""):
        metadata = {
            f"{prefix}pattern_type_method": self.pattern_type_method,
            f"{prefix}shape_zyx": np.asarray(self.shape, dtype=int),
        }
        for key, value in self.pattern_config.items():
            metadata[f"{prefix}pattern_config/{key}"] = value
        return metadata


output_folder = DATA_ROOT / "Data" / "mumax_ck_outputs"
output_folder.mkdir(parents=True, exist_ok=True)
output_path = output_folder / "mumax_ck_simulation.h5"

supportmask = front_aperture_config.create_supportmask(
    output_shape=detectorconfig.detector_layout.detector_shape,
    output_pixel_size=detectorconfig.detector_layout.real_space_resolution,
)
pipeline_aperture_config = {
    "aperture_types": apertures_types,
    "aperture_radii": apertures_radius,
    "aperture_lengths": [0.0] * len(apertures_types),
    "aperture_centers": apertures_centers,
    "aperture_sigmas": apertures_sigma,
    "aperture_angles": apertures_angle,
    "aperture_ellipticities": apertures_ellipticity,
    "aperture_roughnesses": apertures_roughness,
    "aperture_roughness_modes": apertures_roughness_modes,
    "aperture_seeds": apertures_seed,
    "aperture_top_radius_factors": apertures_top_radius_factor,
}

mumax_metadata_config = MumaxPatternMetadata(mumax, magnetic_layer_indices)
canonical_metadata = HologramPipeline.build_precomputed_metadata(
    xray_config=xrayconfig,
    detector_config=detectorconfig,
    beamstop_config=beamstop_config,
    sample_config=sampleconfig,
    magnetic_pattern_config=mumax_metadata_config,
    aperture_config=front_aperture_config,
    illumination_config=illuminationconfig,
    propagator_config=samplepropagationconfig,
    use_roi=True,
    magnetic_pattern_use_roi=False,
    dielectric_tensor_use_roi=True,
    dielectric_tensor_compact=True,
    save_detected_hologram_without_beamstop=False,
)

export_pipeline = HologramPipeline(
    config=HologramPipelineConfig(recipe=recipe, oversampling=oversampling),
    ranges=HologramPipelineRanges(),
    output_path=output_path,
    n_samples=1,
    verbose=False,
)
written_path = export_pipeline.write_precomputed_result(
    hologram_config=hologram_config,
    detector_config=detectorconfig,
    metadata=canonical_metadata,
    aperture_config=pipeline_aperture_config,
    supportmask=supportmask,
    magnetic_pattern_oh=magnetic_pattern_oh,
    overwrite=True,
)
print(f"Wrote canonical pipeline HDF5 file: {written_path}")
